# Resolución orientativa de la práctica de la semana 1
600 usuarios sintéticos. Frecuencia de apertura: número de aperturas en 30 días; funciones usadas: cantidad de funciones distintas; hora central: hora de interacción representativa entre 0 y 24. Para respetar su carácter cíclico se convierte en seno y coseno. Los perfiles del generador no son etiquetas de negocio verdaderas.
Ejecuta desde la carpeta Recursos. La propuesta es una solución posible; compara configuraciones y justifica tus decisiones.

## Datos incluidos en el propio notebook
Ejecuta las celdas en orden dentro de una carpeta de trabajo. La primera genera los CSV sintéticos; no necesitas descargar datos. Puedes conservar esa celda sin ejecutar si quieres analizar tus propias copias de los archivos.

In [ ]:
# Importación de librerías necesarias para el análisis y modelado.
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

In [ ]:
# Generación de los datos sintéticos de la práctica.
rng = np.random.default_rng(42)
filas = []
for i in range(600):
    g = i % 3
    frecuencia = max(1, int(rng.normal([6, 22, 45][g], [2, 6, 9][g])))
    funciones = int(np.clip(round(rng.normal([2, 4, 6][g], 1)), 1, 8))
    hora = float(rng.normal([8, 14, 21][g], 2) % 24)
    filas.append([f"U{i+1:04}", frecuencia, funciones, round(hora, 2)])
columnas = ["usuario_id", "frecuencia_apertura", "funciones_usadas", "hora_central"]
pd.DataFrame(filas, columns=columnas).to_csv("usuarios_app.csv", index=False)

In [ ]:
# Carga de los datos y verificación de su integridad.
df = pd.read_csv("usuarios_app.csv")
assert df.usuario_id.is_unique and not df.isna().any().any()
df

,usuario_id,frecuencia_apertura,funciones_usadas,hora_central
0,U0001,6,1,9.50
1,U0002,27,2,11.40
2,U0003,46,6,20.97
3,U0004,4,3,9.56
4,U0005,22,5,14.94
...,...,...,...,...
595,U0596,29,5,10.82
596,U0597,47,7,17.68
597,U0598,5,2,7.40
598,U0599,10,4,13.35


In [ ]:
# Transformación de la variable hora en dos variables circulares.
hora = 2*np.pi*df.hora_central/24
X = np.column_stack([df.frecuencia_apertura, df.funciones_usadas,
                     np.sin(hora), np.cos(hora)])
X_df = pd.DataFrame(X, columns=["frecuencia_apertura", "funciones_usadas", "sin_hora", "cos_hora"]).head()
X_df

,frecuencia_apertura,funciones_usadas,sin_hora,cos_hora
0,6.0,1.0,0.608761,-0.793353
1,27.0,2.0,0.156434,-0.987688
2,46.0,6.0,-0.712639,0.701531
3,4.0,3.0,0.596225,-0.802817
4,22.0,5.0,-0.695913,-0.718126


In [9]:
# Escalamos las cantidades; mantenemos el bloque horario en el círculo unidad.
X[:, :2] = StandardScaler().fit_transform(X[:, :2]) # :2 indica que solo escalamos las dos primeras columnas, dejando sin cambios las dos últimas (sin_hora y cos_hora).
X_df = pd.DataFrame(X, columns=["frecuencia_apertura", "funciones_usadas", "sin_hora", "cos_hora"]).head()
X_df

,frecuencia_apertura,funciones_usadas,sin_hora,cos_hora
0,-1.030207,-1.608990,0.608761,-0.793353
1,0.201233,-1.066024,0.156434,-0.987688
2,1.315393,1.105841,-0.712639,0.701531
3,-1.147487,-0.523057,0.596225,-0.802817
4,-0.091967,0.562875,-0.695913,-0.718126


In [10]:
# Evaluación de segmentaciones con KMeans y selección de k por silueta.
resultados = []
modelos = {}

for k in range(2, 7):
    m = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    resultados.append((k, m.inertia_, silhouette_score(X, m.labels_)))
    modelos[k] = m

tabla = pd.DataFrame(resultados, columns=["k","inercia","silueta"])
print(tabla.to_string(index=False))

 k    inercia  silueta
 2 730.925860 0.488752
 3 380.741548 0.509590
 4 340.408711 0.410527
 5 303.482978 0.353919
 6 270.776960 0.271348


In [ ]:
k = int(tabla.loc[tabla.silueta.idxmax(), "k"])
df["cluster"] = modelos[k].labels_

print("Candidato por silueta; contrastar con perfiles y propósito:", k)
print(df.groupby("cluster")[["frecuencia_apertura","funciones_usadas"]].agg(["mean","median","count"]))
otra = KMeans(n_clusters=k,n_init=10,random_state=7).fit_predict(X)
print("ARI entre semillas sobre los mismos usuarios:", adjusted_rand_score(df.cluster,otra)) # Medida de similitud entre dos particiones
df.to_csv("resultado_segmentacion.csv",index=False)

Candidato por silueta; contrastar con perfiles y propósito: 3
        frecuencia_apertura              funciones_usadas             
                       mean median count             mean median count
cluster                                                               
0                 44.000000   43.0   199         5.874372    6.0   199
1                  5.683168    5.0   202         1.975248    2.0   202
2                 21.291457   22.0   199         4.070352    4.0   199
ARI entre semillas sobre los mismos usuarios: 1.0


In [12]:
# Comparación gráfica y perfil horario circular.
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].plot(tabla.k,tabla.inercia,"o-");ax[0].set(xlabel="Número de grupos k",ylabel="Inercia",title="Método del codo")
ax[1].plot(tabla.k,tabla.silueta,"o-");ax[1].set(xlabel="Número de grupos k",ylabel="Silueta",title="Separación y cohesión")
fig.suptitle("Figura P1. Comparación de segmentaciones sintéticas")
fig.text(.5,.01,"Fuente: elaboración propia con usuarios_app.csv y semilla 42.",ha="center",fontsize=8)
fig.tight_layout(rect=(0,.04,1,.93));fig.savefig("comparacion_k.png",dpi=160);plt.close(fig)

In [13]:
for grupo,datos in df.groupby("cluster"):
    angulo=2*np.pi*datos.hora_central.to_numpy()/24
    seno,coseno=np.sin(angulo).mean(),np.cos(angulo).mean()
    concentracion=np.hypot(seno,coseno)
    hora_media=(np.arctan2(seno,coseno)%(2*np.pi))*24/(2*np.pi)
    print(f"Grupo {grupo}: hora circular {hora_media:.2f}, concentración {concentracion:.3f}")
    # Una concentración cercana a cero impediría resumir horarios con una sola hora.

Grupo 0: hora circular 20.74, concentración 0.889
Grupo 1: hora circular 8.04, concentración 0.867
Grupo 2: hora circular 14.11, concentración 0.863


## Interpretación orientativa
El candidato con mayor silueta describe una partición geométrica útil para explorar los datos simulados. El codo se lee como una reducción de las mejoras marginales de inercia; no constituye una demostración de un número verdadero de segmentos. Hay que contrastar ambas evidencias con los perfiles y con la estabilidad entre semillas sobre los mismos usuarios.

La hora se interpreta en un círculo. Una media aritmética de 23 y 1 produce 12, que describe mal dos horarios próximos a medianoche. La media circular y su concentración evitan ese error; si la concentración es baja conviene describir la distribución completa.

Los grupos permiten proponer hipótesis sobre frecuencia y variedad de uso. Una acción posible es diseñar mensajes o recorridos de ayuda diferenciados y evaluar su efecto en una prueba posterior. Estos registros no permiten concluir capacidad de pago, rentabilidad ni causalidad. La recomendación de priorizar Producto depende de los supuestos del caso; una alternativa distinta sería defendible con información adicional y un criterio explícito.


## Comparación breve con clustering jerárquico
Reutiliza la misma matriz X y el mismo k de K-Means para aplicar enlace Ward. Completa una tabla de dos filas con silueta y tamaños mínimo y máximo de grupo. Concluye en 80–120 palabras qué método conservarías, relacionando las métricas con los perfiles ya interpretados. Alcance orientativo: 10–15 minutos, una sola configuración adicional, sin nuevo conjunto de datos ni dendrograma obligatorio.

In [ ]:
# Complemento breve: K-Means frente a jerárquico con enlace Ward.
from sklearn.cluster import AgglomerativeClustering
etiquetas_ward = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X)
etiquetas_kmeans = modelos[k].labels_
comparacion = pd.DataFrame([
    {"algoritmo": nombre, "k": len(np.unique(etiquetas)),
     "silueta": silhouette_score(X, etiquetas),
     "tamano_min": int(pd.Series(etiquetas).value_counts().min()),
     "tamano_max": int(pd.Series(etiquetas).value_counts().max())}
    for nombre, etiquetas in [("K-Means", etiquetas_kmeans), ("Jerarquico Ward", etiquetas_ward)]
])
print("Tabla P1. Comparación sobre los mismos 600 usuarios y la misma matriz X")
print(comparacion.round(4).to_string(index=False))
print("Fuente: elaboración propia con usuarios_app.csv; K-Means con semilla 42.")
ari_metodos = adjusted_rand_score(etiquetas_kmeans, etiquetas_ward)
print("ARI entre métodos (acuerdo, no exactitud):", round(ari_metodos, 4))
comparacion.to_csv("resultado_comparacion_algoritmos.csv", index=False)
asignaciones = df[["usuario_id"]].copy()
asignaciones["cluster_kmeans"] = etiquetas_kmeans
asignaciones["cluster_ward"] = etiquetas_ward
asignaciones.to_csv("resultado_asignaciones_algoritmos.csv", index=False)
print("Los identificadores de grupo son arbitrarios y no se comparan por su número.")

Tabla P1. Comparación sobre los mismos 600 usuarios y la misma matriz X
      algoritmo  k  silueta  tamano_min  tamano_max
        K-Means  3   0.5096         199         202
Jerarquico Ward  3   0.5025         193         204
Fuente: elaboración propia con usuarios_app.csv; K-Means con semilla 42.
ARI entre métodos (acuerdo, no exactitud): 0.9412
Los identificadores de grupo son arbitrarios y no se comparan por su número.


## Solución orientativa del complemento
Comparación orientativa: con k = 3, la ejecución adjunta obtiene siluetas de 0,5096 para K-Means y 0,5025 para Ward; los tamaños de grupo varían entre 199 y 202, y entre 193 y 204, respectivamente. El ARI entre métodos es 0,9412: existe mucho acuerdo, pero no identidad. K-Means es una elección defendible por su ligera ventaja geométrica y los perfiles ya examinados. La diferencia no prueba superioridad general; Ward ofrece un contraste útil bajo las mismas condiciones. La conclusión debe conectarse con el propósito de segmentación y con sus limitaciones.